In [ ]:
!pip install -q rfdetr supervision pycocotools optuna
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))  # CHECK: should say True and a GPU name (T4). If False, set Runtime > Change runtime type > T4 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# CHECK: change these 2 paths to where your folders are in Drive
IMG_ROOT = "/content/drive/MyDrive/dataset/images"            # has test/ train/ valid/ inside
LABEL_ROOT = "/content/drive/MyDrive/dataset/rf-detr_labels"  # has test_annotations.json etc

DATASET_DIR = "/content/calamansi_coco"   # rebuilt on Colab's local disk (faster training)

# CHECK: True = quick 3 epoch test. Set to False for the real run
SMOKE_TEST = True

if SMOKE_TEST:
    EPOCHS = 3
    OUTPUT_DIR = "/content/smoke_test_run"                       # throwaway folder, not saved to Drive
else:
    EPOCHS = 50
    OUTPUT_DIR = "/content/drive/MyDrive/calamansi/rfdetr_run1"  # real run, saved to Drive

# CHECK: after Optuna, replace these with the best params it prints
HP = {"lr": 1e-4, "batch_size": 4, "grad_accum_steps": 4}

In [ ]:
import json, glob
from collections import Counter

for split in ["train", "valid", "test"]:
    matches = glob.glob(f"{LABEL_ROOT}/{split}_annotations*.json")
    print(split, "->", matches)  # CHECK: exactly 1 file per split. If empty, the file name or LABEL_ROOT is wrong
    coco = json.load(open(matches[0]))
    print("  images:", len(coco["images"]))                                # CHECK: roughly 73 / 16 / 16
    print("  annotations:", len(coco["annotations"]))                      # CHECK: ~30 per image
    print("  categories:", [c["name"] for c in coco["categories"]])        # CHECK: same 4 classes in all 3 splits
    print("  per class:", dict(Counter(a["category_id"] for a in coco["annotations"])))
    print("  has bbox:", all("bbox" in a for a in coco["annotations"]))    # CHECK: must be True

In [ ]:
import json, os, glob, shutil

if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)  # start clean so old files don't mix in

for split in ["train", "valid", "test"]:
    out = os.path.join(DATASET_DIR, split)
    os.makedirs(out, exist_ok=True)

    coco = json.load(open(glob.glob(f"{LABEL_ROOT}/{split}_annotations*.json")[0]))

    # clean up annotations: drop polygons (we only use boxes) and make sure iscrowd exists
    for a in coco["annotations"]:
        a.pop("segmentation", None)
        a.setdefault("iscrowd", 0)

    # copy the images listed in the JSON
    missing = []
    for img in coco["images"]:
        img["file_name"] = os.path.basename(img["file_name"])  # in case the JSON has folder prefixes
        src = os.path.join(IMG_ROOT, split, img["file_name"])
        if os.path.exists(src):
            shutil.copy(src, out)
        else:
            missing.append(img["file_name"])

    json.dump(coco, open(os.path.join(out, "_annotations.coco.json"), "w"))
    # CHECK: "missing" should be 0. If not, the file names in the JSON don't match the files in your images folder
    print(split, "| images copied:", len(coco["images"]) - len(missing), "| missing:", len(missing))
    if missing:
        print("  e.g.", missing[:3])

In [ ]:
import time, gc
import supervision as sv
from PIL import Image
from rfdetr import RFDETRBase

def evaluate(checkpoint_path, split="valid"):
    model = RFDETRBase(pretrain_weights=checkpoint_path)
    model.optimize_for_inference()

    ds = sv.DetectionDataset.from_coco(
        images_directory_path=f"{DATASET_DIR}/{split}",
        annotations_path=f"{DATASET_DIR}/{split}/_annotations.coco.json",
    )
    name_to_idx = {n: i for i, n in enumerate(ds.classes)}  # map class ids by name so they line up

    preds, targets, times = [], [], []
    for path, _, gt in ds:
        img = Image.open(path)
        t0 = time.time()
        det = model.predict(img, threshold=0.01)
        times.append(time.time() - t0)
        det.class_id = [name_to_idx[model.class_names[c]] for c in det.class_id]
        preds.append(det)
        targets.append(gt)

    m = sv.metrics.MeanAveragePrecision().update(preds, targets).compute()
    fps = 1 / (sum(times[5:]) / max(len(times[5:]), 1))  # skips the first few warmup frames
    del model
    gc.collect(); torch.cuda.empty_cache()
    return m.map50, m.map50_95, fps, preds, targets

In [ ]:
from rfdetr import RFDETRBase

model = RFDETRBase()
model.train(
    dataset_dir=DATASET_DIR,
    output_dir=OUTPUT_DIR,            # CHECK: should be /content/smoke_test_run while SMOKE_TEST = True
    epochs=EPOCHS,                    # CHECK: should be 3 during smoke test
    batch_size=HP["batch_size"],      # if CUDA out of memory: set to 2 and grad_accum_steps to 8 in Cell 1
    grad_accum_steps=HP["grad_accum_steps"],
    lr=HP["lr"],
    early_stopping=True,
    early_stopping_patience=10,
    aug_config={},                    # no augmentation for now. If this line errors, delete it
)

!ls {OUTPUT_DIR}                      # CHECK: you should see checkpoint_best_total.pth in the list

In [ ]:
import glob

ckpt = f"{OUTPUT_DIR}/checkpoint_best_total.pth"   # CHECK: if the file name differs, use the one from !ls above

map50, map5095, fps, _, _ = evaluate(ckpt, "valid")
print(f"valid mAP@50: {map50:.3f}  mAP@50:95: {map5095:.3f}  FPS: {fps:.1f}")  # CHECK: mAP@50 > 0 means the pipeline works

model = RFDETRBase(pretrain_weights=ckpt)
paths = sorted(glob.glob(f"{DATASET_DIR}/valid/*.jpg") + glob.glob(f"{DATASET_DIR}/valid/*.png"))[:3]
for p in paths:
    img = Image.open(p)
    det = model.predict(img, threshold=0.4)
    labels = [f"{model.class_names[c]} {s:.2f}" for c, s in zip(det.class_id, det.confidence)]
    out = sv.BoxAnnotator().annotate(img.copy(), det)
    out = sv.LabelAnnotator().annotate(out, det, labels)
    print(p, "->", len(det), "detections")   # CHECK: around 30 per image. 0 or 100+ means something is off
    sv.plot_image(out)                        # CHECK: boxes should sit on the fruits

In [ ]:
import optuna

# each trial = one training run, so keep it short
N_TRIALS = 12        # CHECK: 10 to 15 is a good range
TRIAL_EPOCHS = 15

def objective(trial):
    lr = trial.suggest_float("lr", 3e-5, 3e-4, log=True)
    bs = trial.suggest_categorical("batch_size", [2, 4, 8])
    accum = 16 // bs                                    # keeps effective batch size at 16

    out = f"/content/optuna_trials/trial_{trial.number}"
    model = RFDETRBase()
    model.train(
        dataset_dir=DATASET_DIR, output_dir=out,
        epochs=TRIAL_EPOCHS, batch_size=bs, grad_accum_steps=accum, lr=lr,
        early_stopping=True, early_stopping_patience=5,
        aug_config={},
    )
    del model
    gc.collect(); torch.cuda.empty_cache()

    # score on VALIDATION only, never the test set
    map50, _, _, _, _ = evaluate(f"{out}/checkpoint_best_total.pth", "valid")
    return map50

# saved to Drive so you can resume if Colab disconnects
study = optuna.create_study(
    direction="maximize",
    study_name="rfdetr_calamansi",
    storage="sqlite:////content/drive/MyDrive/calamansi/optuna.db",
    load_if_exists=True,
)
study.optimize(objective, n_trials=N_TRIALS)

print("best params:", study.best_params)
print("best valid mAP@50:", study.best_value)

In [ ]:
ckpt = f"{OUTPUT_DIR}/checkpoint_best_total.pth"   # CHECK: SMOKE_TEST must be False and this should point to the Drive folder

map50, map5095, fps, test_preds, test_targets = evaluate(ckpt, "test")
print(f"TEST  mAP@50: {map50:.3f}  mAP@50:95: {map5095:.3f}  FPS: {fps:.1f}")